<a href="https://colab.research.google.com/github/PRAGYAARAY/ZenTrack/blob/main/nlp_sentiment_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline
import os

# Global pipeline instance — loaded once, reused across requests
_sentiment_pipeline = None

def get_sentiment_pipeline():
    """
    Load an Emotion-specific DistilBERT model (Lazy Loading).
    Downloads and caches the weights on the very first API call.
    """
    global _sentiment_pipeline
    if _sentiment_pipeline is None:
        print('==================================================')
        print('🚀 Loading GoEmotions/Emotion Analysis NLP Engine...')
        print('==================================================')

        _sentiment_pipeline = pipeline(
            task='text-classification',
            model='bhadresh-savani/distilbert-base-uncased-emotion',
            return_all_scores=True,  # Returns probability arrays for ALL emotions
            truncation=True,
            max_length=512
        )
        print('🎉 Emotion processing engine loaded successfully!')
    return _sentiment_pipeline
text=input("Tell us how you feel:")
def analyse_sentiment(text):
    """
    Analyzes user diary text for clinical/burnout emotional patterns.
    Auto-flattens nested list payloads to prevent silent fallback loops.
    """
    if not text or len(text.strip()) < 5:
        return 0.0, 'neutral'

    try:
        pipe = get_sentiment_pipeline()
        text = text[:512]
        raw_output = pipe(text)

        # 🛠️ AUTO-FLATTEN LAYER: Strip any unexpected nested list arrays
        # Handles both [[{...}]] and [{...}] formats seamlessly
        while isinstance(raw_output, list) and len(raw_output) > 0 and isinstance(raw_output[0], list):
            raw_output = raw_output[0]

        if isinstance(raw_output, list) and len(raw_output) > 0 and isinstance(raw_output[0], dict):
            raw_list = raw_output
        else:
            raise ValueError(f"Unexpected pipeline structure output format: {type(raw_output)}")

        # Map list structure into an easily accessible dictionary
        emotion_scores = {res['label']: res['score'] for res in raw_list}

        # Extract individual metrics vital to tracking burnout
        sadness = emotion_scores.get('sadness', 0.0)
        fear    = emotion_scores.get('fear', 0.0)     # Maps to anxiety
        anger   = emotion_scores.get('anger', 0.0)    # Maps to frustration
        joy     = emotion_scores.get('joy', 0.0)

        # Calculate a combined distress index
        distress_weight = (sadness * 0.45) + (fear * 0.35) + (anger * 0.20)

        print(f"📊 Debug Metrics -> Sadness: {sadness:.2f}, Fear: {fear:.2f}, Anger: {anger:.2f}, Joy: {joy:.2f}")
        print(f"📉 Combined Distress Calculated: {distress_weight:.2f}")

        # Evaluation Logic Matrix
        if joy > 0.50:
            score = round(joy, 4)
            display_label = 'positive'

        elif distress_weight > 0.20:  # Lowered slightly to capture mid-tier burnout text signals
            score = round(-distress_weight, 4)

            # Dynamically pull the exact leading negative emotion flag
            negative_candidates = ['sadness', 'fear', 'anger']
            display_label = max(negative_candidates, key=lambda k: emotion_scores.get(k, 0.0))

        else:
            score = 0.0
            display_label = 'neutral'

        return score, display_label

    except Exception as e:
        # If an error happens, print it directly to the terminal so we can see it!
        print(f'❌ NLP Engine Runtime Error: {e}')
        return 0.0, 'neutral'
print(analyse_sentiment(text))

Tell us how you feel:i hate everything
🚀 Loading GoEmotions/Emotion Analysis NLP Engine...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

🎉 Emotion processing engine loaded successfully!
📊 Debug Metrics -> Sadness: 0.00, Fear: 0.00, Anger: 0.98, Joy: 0.00
📉 Combined Distress Calculated: 0.20
(0.0, 'neutral')


In [ ]:
from transformers import pipeline
import os

# Global pipeline instance — loaded once, reused across requests
_sentiment_pipeline = None

def get_sentiment_pipeline():
    """
    Load an Emotion-specific DistilBERT model (Lazy Loading).
    Downloads and caches the weights on the very first API call.
    """
    global _sentiment_pipeline
    if _sentiment_pipeline is None:
        print('==================================================')
        print('🚀 Loading GoEmotions/Emotion Analysis NLP Engine...')
        print('==================================================')

        _sentiment_pipeline = pipeline(
            task='text-classification',
            model='bhadresh-savani/distilbert-base-uncased-emotion',
            return_all_scores=True,  # Returns probability arrays for ALL emotions
            truncation=True,
            max_length=512
        )
        print('🎉 Emotion processing engine loaded successfully!')
    return _sentiment_pipeline
text=input("tell bus how you feel:")
def analyse_sentiment(text):
    """
    Analyzes user diary text and extracts the absolute strongest emotion detected.

    Returns:
        score (float): The confidence score of that specific dominant emotion (0.0 to 1.0)
        display_label (str): The name of the strongest emotion (e.g., 'sadness', 'fear', 'joy')
    """
    if not text or len(text.strip()) < 5:
        return 0.0, 'neutral'

    try:
        pipe = get_sentiment_pipeline()
        text = text[:512]
        raw_output = pipe(text)

        # Auto-flatten layer to handle variant array dimensions from transformers package
        while isinstance(raw_output, list) and len(raw_output) > 0 and isinstance(raw_output[0], list):
            raw_output = raw_output[0]

        if not isinstance(raw_output, list) or len(raw_output) == 0:
            raise ValueError("Empty or invalid output received from pipeline engine.")

        # Map list structure into a flat dictionary
        # Example: {'sadness': 0.12, 'joy': 0.02, 'fear': 0.84, ...}
        emotion_scores = {res['label']: float(res['score']) for res in raw_output}

        # Find the emotion key with the absolute highest probability value
        strongest_emotion = max(emotion_scores, key=emotion_scores.get)
        confidence_score = round(emotion_scores[strongest_emotion], 4)

        # Print debug statistics to terminal window
        print("\n--- 🧠 NLP Emotion Analysis Diagnostics ---")
        for emotion, val in emotion_scores.items():
            print(f" * {emotion.capitalize()}: {val:.4f}")
        print(f"🏆 DOMINANT SIGNAL: {strongest_emotion.upper()} ({confidence_score})")
        print("-------------------------------------------\n")

        # Return the confidence value and the string label of that exact winning emotion
        return confidence_score, strongest_emotion

    except Exception as e:
        print(f'❌ NLP Engine Runtime Error: {e}')
        # Fallback safeguard pattern
        return 0.0, 'neutral'


print(analyse_sentiment(text))

tell bus how you feel:disappointed
🚀 Loading GoEmotions/Emotion Analysis NLP Engine...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

🎉 Emotion processing engine loaded successfully!

--- 🧠 NLP Emotion Analysis Diagnostics ---
 * Sadness: 0.9797
🏆 DOMINANT SIGNAL: SADNESS (0.9797)
-------------------------------------------

(0.9797, 'sadness')


In [ ]:
from transformers import pipeline
import os

# Global pipeline instance — loaded once, reused across requests
_sentiment_pipeline = None

def get_sentiment_pipeline():
    """
    Load an Emotion-specific DistilBERT model (Lazy Loading).
    Downloads and caches the weights on the very first API call.
    """
    global _sentiment_pipeline
    if _sentiment_pipeline is None:
        print('==================================================')
        print('🚀 Loading GoEmotions/Emotion Analysis NLP Engine...')
        print('==================================================')

        _sentiment_pipeline = pipeline(
            task='text-classification',
            model='bhadresh-savani/distilbert-base-uncased-emotion',
            return_all_scores=True,  # Returns probability arrays for ALL emotions
            truncation=True,
            max_length=512
        )
        print('🎉 Emotion processing engine loaded successfully!')
    return _sentiment_pipeline

text = input("Tell us how you feel (e.g., 'I am very happy today' or 'I feel so alone'):")

def analyse_sentiment(text):
    """
    Analyzes user text and extracts overall sentiment (positive, negative, neutral)
    and identifies the strongest contributing emotion.

    Returns:
        score (float): The confidence score of the primary sentiment (positive values for positive, negative for negative).
        display_label (str): The primary sentiment ('positive', 'negative', 'neutral') or the strongest emotion label.
    """
    if not text or len(text.strip()) < 5:
        return 0.0, 'neutral'

    try:
        pipe = get_sentiment_pipeline()
        text = text[:512]
        raw_output = pipe(text)

        # Auto-flatten layer to handle variant array dimensions from transformers package
        while isinstance(raw_output, list) and len(raw_output) > 0 and isinstance(raw_output[0], list):
            raw_output = raw_output[0]

        if not isinstance(raw_output, list) or len(raw_output) == 0:
            raise ValueError("Empty or invalid output received from pipeline engine.")

        emotion_scores = {res['label']: float(res['score']) for res in raw_output}

        # Define emotion categories based on the model's known labels
        positive_labels = ['admiration', 'amusement', 'approval', 'caring', 'desire', 'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief', 'surprise']
        negative_labels = ['anger', 'annoyance', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'fear', 'grief', 'nervousness', 'remorse', 'sadness']
        neutral_labels = ['confusion', 'curiosity', 'neutral', 'realization'] # Includes some potentially ambiguous but often non-valence labels

        # Calculate composite scores for overall positive and negative sentiment
        total_positive_score = sum(emotion_scores.get(label, 0.0) for label in positive_labels)
        total_negative_score = sum(emotion_scores.get(label, 0.0) for label in negative_labels)

        # Find the single strongest emotion for detailed diagnostics
        strongest_emotion_label = max(emotion_scores, key=emotion_scores.get)
        strongest_emotion_confidence = emotion_scores[strongest_emotion_label]

        # Print debug statistics
        print("\n--- 🧠 NLP Emotion Analysis Diagnostics ---")
        for emotion, val in emotion_scores.items():
            print(f" * {emotion.capitalize()}: {val:.4f}")
        print(f"Total Positive Score: {total_positive_score:.4f}")
        print(f"Total Negative Score: {total_negative_score:.4f}")
        print(f"🏆 STRONGEST INDIVIDUAL EMOTION: {strongest_emotion_label.upper()} ({strongest_emotion_confidence:.4f})")
        print("-------------------------------------------\n")

        # Determine the overall sentiment based on a more human-like hierarchy
        # Prioritize strong negative signals
        if total_negative_score > 0.40: # Threshold for significant negative sentiment
            score = -total_negative_score
            display_label = 'negative' # Can refine to strongest negative emotion if needed
            # If we want the specific leading negative emotion, use: max(negative_labels, key=lambda k: emotion_scores.get(k, 0.0))
        elif total_positive_score > 0.40: # Threshold for significant positive sentiment
            score = total_positive_score
            display_label = 'positive'
        elif strongest_emotion_label in neutral_labels and strongest_emotion_confidence > 0.5: # Consider strong neutral signals
            score = strongest_emotion_confidence
            display_label = 'neutral'
        else:
            # Default to neutral if no strong signals, or use the single strongest emotion if it's moderate
            score = 0.0
            display_label = 'neutral'
            if strongest_emotion_confidence > 0.2: # If a moderate emotion is detected, report it
                display_label = strongest_emotion_label

        return score, display_label

    except Exception as e:
        print(f'❌ NLP Engine Runtime Error: {e}')
        # Fallback safeguard pattern
        return 0.0, 'neutral'

print(analyse_sentiment(text))

Tell us how you feel (e.g., 'I am very happy today' or 'I feel so alone'):stressed
🚀 Loading GoEmotions/Emotion Analysis NLP Engine...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

🎉 Emotion processing engine loaded successfully!

--- 🧠 NLP Emotion Analysis Diagnostics ---
 * Anger: 0.9568
Total Positive Score: 0.0000
Total Negative Score: 0.9568
🏆 STRONGEST INDIVIDUAL EMOTION: ANGER (0.9568)
-------------------------------------------

(-0.9568266272544861, 'negative')
